# P1.01 · Política de decisão por lucro esperado
**Motor de concessão de crédito E2E** · camada **L3**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1thirteeng3/ds-ai-cookbook/blob/colab/pt/projects/P1-credit/P1.01-e2e-credit-decision-engine/notebooks/32-decision-policy.ipynb)

| Nível | Runtime Colab | Tempo (smoke / full) | Documento |
|---|---|---|---|
| 🟠 L3 | cpu | 1 / 3 min | [index.pt.md](../index.pt.md) |

> Transforma PD em aprovar/recusar maximizando lucro, com restrição de taxa de aprovação, e compara com a regra de 70% da L1: quem entra, quem sai e quanto isso vale.

## 0. Parâmetros

`RUN_MODE="smoke"` usa 10% da base (ou o gêmeo sintético com `OFFLINE=True`) e roda em minutos. `RUN_MODE="full"` usa a base inteira e produz os resultados oficiais. `OFFLINE=True` só existe para testes e CI: nunca tire conclusões dele.

In [ ]:
# Parameters (papermill overrides this cell)
RUN_MODE = "smoke"   # "smoke" | "full"
SEED = 42
DATA_DIR = "data"    # Colab + Drive: "/content/drive/MyDrive/cookbook-data"
OFFLINE = False      # True: synthetic twin (tests/CI). Never for full results.
SAMPLE_FRAC = 0.1    # smoke only
N_BOOT = 200         # bootstrap resamples for metric uncertainty
MARGIN = 0.10         # net lifetime margin per unit of credit on a good loan (assumption)
LGD = 0.60            # loss given default on a bad loan (assumption)
MIN_APPROVAL = 0.60   # business constraint: approve at least 60%
L1_APPROVAL = 0.70    # the L1 rule we compare against

## 1. Setup

No Colab, instala o pacote `cookbook` do repositório privado (segredo `GH_TOKEN`) e as versões exatas usadas nos resultados. NumPy, pandas e scikit-learn ficam com as versões do Colab e são registrados no arquivo de resultados. Se a instalação atualizar algum deles, use *Ambiente de execução > Reiniciar sessão* e rode a partir da seção 0. Para baixar o Home Credit, crie o segredo `KAGGLE_API_TOKEN` (ou `KAGGLE_USERNAME` e `KAGGLE_KEY`) e aceite as regras da competição no kaggle.com com a mesma conta.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = {
    "optbinning": "1.0.0",
    "lightgbm": "4.7.0",
    "optuna": "5.0.0",
    "mlflow": "3.18.0",
    "fastapi": "0.162.0",
    "httpx": "0.28.1",
    "polars": "2.1.0",
    "pyarrow": "26.0.1",
    "kaggle": "2.2.5",
    "joblib": "1.6.0",
    "matplotlib": "3.11.2",
}

if IN_COLAB and not sys.modules.get("cookbook"):
    from google.colab import userdata

    token = userdata.get("GH_TOKEN")
    pins = " ".join(f"{k}=={v}" for k, v in PACKAGES.items())
    %pip install -q uv
    !uv pip install --system -q {pins} "cookbook[credit,kaggle] @ git+https://{token}@github.com/1thirteeng3/cookbokk.git@main"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from cookbook import credit as cr
from cookbook import home_credit as hc
from cookbook.data import sample
from cookbook.env import colab_secrets_to_env, hardware_info, quiet_logs, set_seeds, versions
from cookbook.results import Timer, save_result

quiet_logs()  # before heavy imports: silences optional-solver probing
import json

import matplotlib.pyplot as plt

set_seeds(SEED)
colab_secrets_to_env()
if RUN_MODE == "full" and OFFLINE:
    raise ValueError("RUN_MODE='full' needs real data (OFFLINE=False).")

SOURCE = "home-credit-synthetic" if OFFLINE else "home-credit"
ARTIFACTS = Path(DATA_DIR) / "artifacts" / "P1.01" / RUN_MODE
ARTIFACTS.mkdir(parents=True, exist_ok=True)
HW = hardware_info()
VERSIONS = versions([*PACKAGES, "numpy", "pandas", "scikit-learn"])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)
print(SOURCE, RUN_MODE, ARTIFACTS, HW)

In [ ]:
missing = [f for f in ["l1_scores.parquet", "l3_predictions.parquet"] if not (ARTIFACTS / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {ARTIFACTS}. Run notebooks 10 and 31 first "
        f"with the same RUN_MODE and DATA_DIR. / Rode 10 e 31 antes."
    )

## 2. Dados e premissas econômicas

Usamos os PDs de validação e teste da L1 (scorecard) e da L3 (GBM calibrado) para os mesmos clientes. A conta econômica é deliberadamente simples e **as premissas são parâmetros**: troque pelos números da sua carteira.

- Exposição (EAD) = `AMT_CREDIT`.
- Empréstimo bom rende `MARGIN × EAD` ao longo da vida; empréstimo ruim perde `LGD × EAD`.
- Lucro esperado de aprovar: $(1-PD)\cdot m \cdot EAD - PD \cdot LGD \cdot EAD$, positivo quando $PD < m / (m + LGD)$.

In [ ]:
l1 = pd.read_parquet(ARTIFACTS / "l1_scores.parquet")
l3 = pd.read_parquet(ARTIFACTS / "l3_predictions.parquet")
df = l3.merge(l1[[hc.ID, "pd_l1"]], on=hc.ID, how="inner")
valid, test = df[df["SPLIT"] == "valid"], df[df["SPLIT"] == "test"]
DATASET = json.loads((ARTIFACTS / "feature_meta.json").read_text())["dataset"]
BREAKEVEN_PD = MARGIN / (MARGIN + LGD)
print(f"break-even PD = {BREAKEVEN_PD:.3f}; valid={len(valid)}, test={len(test)}")

## 3. Solução

### 3.1 Curvas de política na validação

Para cada ponto de corte em PD: taxa de aprovação, inadimplência dos aprovados e lucro realizado. O corte é escolhido **na validação** e só depois aplicado ao teste.

In [ ]:
def curve(frame, col):
    return cr.policy_curve(frame[hc.TARGET], frame[col], frame["AMT_CREDIT"], MARGIN, LGD, n_points=201)

def best_threshold(frame, col, min_approval=MIN_APPROVAL):
    c = curve(frame, col)
    feasible = c[c["approval_rate"] >= min_approval]
    return float(feasible.loc[feasible["profit"].idxmax(), "pd_threshold"])

def threshold_for_rate(frame, col, rate):
    return float(np.quantile(frame[col], rate))

THRESHOLDS = {
    "L1 · approve 70%": ("pd_l1", threshold_for_rate(valid, "pd_l1", L1_APPROVAL)),
    "L1 · max profit": ("pd_l1", best_threshold(valid, "pd_l1")),
    "L3 · approve 70%": ("pd_l3", threshold_for_rate(valid, "pd_l3", L1_APPROVAL)),
    "L3 · max profit": ("pd_l3", best_threshold(valid, "pd_l3")),
    "L3 · break-even PD": ("pd_l3", BREAKEVEN_PD),
}
pd.DataFrame({k: {"score": v[0], "pd_threshold": v[1]} for k, v in THRESHOLDS.items()}).T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for col, label in (("pd_l1", "L1 scorecard"), ("pd_l3", "L3 GBM")):
    c = curve(valid, col)
    ax[0].plot(c["approval_rate"], c["profit"] / len(valid), label=label)
    ax[1].plot(c["approval_rate"], c["bad_rate_approved"], label=label)
ax[0].axvline(MIN_APPROVAL, ls="--", c="grey"); ax[0].set_title("profit per application (valid)")
ax[1].set_title("bad rate among approved (valid)")
for a in ax:
    a.set_xlabel("approval rate"); a.legend()
plt.tight_layout(); plt.show()

## 4. Validação no teste

In [ ]:
def evaluate(frame, col, threshold):
    approved = frame[col].to_numpy() <= threshold
    y = frame[hc.TARGET].to_numpy()
    profit = cr.realized_profit(y, frame["AMT_CREDIT"], approved, MARGIN, LGD)
    return {"approval_rate": approved.mean(), "bad_rate_approved": y[approved].mean(),
            "profit_per_app": profit / len(frame), "approved": approved}

RESULTS = {name: evaluate(test, col, t) for name, (col, t) in THRESHOLDS.items()}
summary = pd.DataFrame({k: {m: v[m] for m in ("approval_rate", "bad_rate_approved", "profit_per_app")}
                        for k, v in RESULTS.items()}).T
summary

### 4.1 Análise de troca (*swap set*)

Com a **mesma taxa de aprovação** (70%), quem a L3 aprova e a L1 recusava (*swap-in*), e o contrário (*swap-out*). Como o volume é igual, a diferença isola o ganho do **modelo**: se a L3 ordena melhor, os *swap-ins* têm inadimplência menor que os *swap-outs*. É a tabela que o comitê de crédito costuma pedir. O ganho da **política** (aprovar por lucro) aparece na tabela anterior.

In [ ]:
a1, a3 = RESULTS["L1 · approve 70%"]["approved"], RESULTS["L3 · approve 70%"]["approved"]
y_t = test[hc.TARGET].to_numpy()
groups = {"approved by both": a1 & a3, "swap-in (L3 only)": ~a1 & a3,
          "swap-out (L1 only)": a1 & ~a3, "declined by both": ~a1 & ~a3}
pd.DataFrame({k: {"share": m.mean(), "bad_rate": y_t[m].mean() if m.any() else np.nan} for k, m in groups.items()}).T

## 5. Eixos de variação

### 5.1 Sensibilidade às premissas

O corte ótimo depende de margem e LGD. A tabela refaz a escolha (na validação) para combinações plausíveis e mostra a taxa de aprovação resultante no teste.

In [ ]:
rows = []
for margin in (0.06, 0.10, 0.15):
    for lgd in (0.45, 0.60, 0.75):
        c = cr.policy_curve(valid[hc.TARGET], valid["pd_l3"], valid["AMT_CREDIT"], margin, lgd, n_points=201)
        t = float(c.loc[c["profit"].idxmax(), "pd_threshold"])
        approved = test["pd_l3"].to_numpy() <= t
        rows.append({"margin": margin, "lgd": lgd, "pd_threshold": t, "test_approval": approved.mean(),
                     "test_profit_per_app": cr.realized_profit(y_t, test["AMT_CREDIT"], approved, margin, lgd) / len(test)})
pd.DataFrame(rows).pivot(index="margin", columns="lgd", values="test_approval")

## 6. Armadilhas demonstradas

### 6.1 Escolher o corte no teste

**Sintoma:** lucro de teste otimista. **Correção:** escolher na validação, medir no teste.

### 6.2 Usar score não calibrado na conta de lucro

**Sintoma:** o corte de *break-even* aprova gente demais ou de menos. **Causa:** a fórmula supõe que PD é probabilidade de verdade. **Correção:** calibrar (notebook 31) ou escolher o corte empiricamente pela curva.

In [ ]:
t_on_test = best_threshold(test, "pd_l3")
rank_score = test["raw_l3"].rank(pct=True)   # a rank score in [0, 1]: orders well, means nothing as a PD
pd.DataFrame({
    "profit_per_app": {
        "threshold from valid (correct)": RESULTS["L3 · max profit"]["profit_per_app"],
        "threshold tuned on test (leak)": evaluate(test, "pd_l3", t_on_test)["profit_per_app"],
        "break-even on calibrated PD": RESULTS["L3 · break-even PD"]["profit_per_app"],
        "break-even on a rank score (wrong)": evaluate(test.assign(r=rank_score), "r", BREAKEVEN_PD)["profit_per_app"],
    }
})

## 7. Resultados

Grava a política escolhida em `policy.json` (a API do notebook 33 lê daqui) e as métricas de cada política.

In [ ]:
policy = {"score": "pd_l3", "threshold_pd": THRESHOLDS["L3 · max profit"][1], "margin": MARGIN, "lgd": LGD,
          "min_approval": MIN_APPROVAL, "chosen_on": "valid", "run_mode": RUN_MODE}
(ARTIFACTS / "policy.json").write_text(json.dumps(policy, indent=2))
for name, res in RESULTS.items():
    save_result(
        recipe_id="P1.01", notebook="32-decision-policy", stack="profit-policy", variant=name,
        run_mode=RUN_MODE, hardware=HW, versions=VERSIONS, dataset=DATASET,
        params={"margin": MARGIN, "lgd": LGD, "min_approval": MIN_APPROVAL, "threshold_pd": THRESHOLDS[name][1]},
        metrics={m: {"mean": round(float(res[m]), 6), "std": None, "n": 1}
                 for m in ("approval_rate", "bad_rate_approved", "profit_per_app")},
        time_s={"train": 0.0, "inference_per_1k": None},
    )
policy

## 8. Conclusão

- A decisão passa a depender do valor de cada empréstimo, não só do ranking.
- A análise de troca mostra quem muda de decisão e com que risco: é o argumento para o comitê.
- **Próximo passo:** [33 · API de decisão](33-serving-api.ipynb).